# Théminettes - consolidation des chroniques

Une seule sonde : **CTD** (Diver autonome : niveau, conductivité, température).
Pas de choix de sonde, pas de fusion. Les points de contrôle terrain recalent la chronique.

Les fonctions communes sont dans la librairie `ouysse`. Ce notebook ne garde que
ce qui est propre à la station : chemins, mesures écartées, réglages du filtre.

## 1. Imports

In [ ]:
import os
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ouysse
from ouysse import *          # fonctions communes a toutes les stations

print("ouysse-hydro", ouysse.__version__)

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Théminettes\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Pluie_BV_Ouysse.csv"

UTC_CTD_PATH     = os.path.join(BASE, "UTC_CTD.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, r"Théminettes_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, r"Théminettes_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, r"Graphes.svg")

PREFIXE_CTD = "Théminettes"
BARO_COL    = "Patm Thémines [hPa]"
COL_UTC     = "UTC fichier"     # en-tête de la table UTC
PAS         = "1h"

PARAMETRES = ["Niveau_(cm)", "Conductivité", "Température"]

## 3. CTD : lecture, UTC et compensation barométrique

`lire_CTD` encaisse les pièges du format Diver (en-tête à une ligne variable, pied
`END OF DATA`, virgules décimales, mS/cm ou µS/cm).

In [ ]:
metadata = pd.read_excel(UTC_CTD_PATH)
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = en_utc(lire_CTD(nom, CTD_PATH, PAS), nom,
                               metadata, col_utc=COL_UTC)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Cond_(µS/cm)": "Conductivité", "Température[°C]": "Température"})
    morceaux.append(m[["Date/time"] + PARAMETRES])
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s), {len(merge_ctd_df)} enregistrements.")

## 4. Assemblage sur la grille horaire

Pas d'ancienne chronique à raccorder pour cette station.

In [ ]:
full_data = sur_grille([empiler([merge_ctd_df], PARAMETRES)], PAS)

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data, {**GAMMES, "cond": (50, 5000), "niveau": (15, 1000)})

full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nfichier fusionné : {SORTIE_CONSOLIDE}")

## 5. Corrections capteur

`VOIES_ECARTEES` met des mesures à l'écart. Il n'y a pas de sonde de secours ici :
la lacune reste, et l'interpolation ne comblera pas plus de 12 h.

In [ ]:
#: (début, fin, colonne, motif) : mesures mises à l'écart.
VOIES_ECARTEES = [
]

print("Voies écartées :")
full_data = ecarter(full_data, VOIES_ECARTEES)

## 6. Calage sur les points de contrôle

Le décalage mesuré à chaque point est appliqué vers l'aval, en cascade.
Un point à plus d'une heure de toute mesure est ignoré, avec un message.

In [ ]:
#: (grandeur, fichier, colonne lue dans le fichier) : points de contrôle terrain.
#: La colonne `Correction` du fichier décide : seuls les "Oui" sont appliqués.
POINTS = [
    ("Conductivité", os.path.join(BASE, r"punctual_measurements_conducti.xlsx"), "Conductivité"),
    ("Niveau_(cm)", os.path.join(BASE, r"punctual_measurements.xlsx"), "Hauteur (cm)"),
]

for grandeur, chemin, col in POINTS:
    points = lire_points(chemin)
    print(f"{grandeur} :")
    avant = full_data[grandeur]
    full_data[grandeur] = caler(avant, points, col)
    graphe([(avant, "avant calage", "darkorange"),
            (full_data[grandeur], "calée", "black")],
           titre=grandeur, ylab=grandeur, points=points, col_point=col)

## 7. Filtre IQR et lissage

In [ ]:
FENETRE_IQR, K_IQR = "24h", 0.1   # k = 0 : pas de filtre
LISSAGE_H = 6                     # 0 = pas de lissage ; sinon médiane glissante, en heures

avant = full_data["Conductivité"]
full_data["Conductivité"] = filtre_iqr(avant, FENETRE_IQR, K_IQR, lissage_h=LISSAGE_H)
full_data["Conductivité_Moyenne_Mobile"] = full_data["Conductivité"].rolling("6h", center=True).mean()

graphe([(avant, "avant IQR et lissage", "darkorange"),
        (full_data["Conductivité"], "après IQR et lissage", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)")

## 8. Cote NGF, interpolation et statuts

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur
est mesurée, interpolée ou manquante.

In [ ]:
NIVEAU_NGF = None       # cote du zéro de l'échelle, None si elle n'est pas connue
MAX_TROU_H = 12

full_data = interpoler_avec_statut(full_data, PARAMETRES, MAX_TROU_H, PAS)

if NIVEAU_NGF is not None:
    full_data["Niveau_(mNGF)"] = NIVEAU_NGF + full_data["Niveau_(cm)"] / 100
    full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
    print(f"Zéro de l'échelle à {NIVEAU_NGF:.4f} m NGF")
display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 9. Sauvegarde et graphe de synthèse

In [ ]:
finaux = [c for c in PARAMETRES + ["Niveau_(mNGF)"] if c in full_data]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}") if c in full_data]
sortie = full_data[colonnes].copy()
sortie.attrs["ouysse"] = ouysse.__version__
sortie.to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(sortie)} pas x {len(colonnes)} colonnes "
      f"(ouysse-hydro {ouysse.__version__})")

graphe_synthese(full_data, "Niveau_(cm)", "Niveau (cm)", pluie=PLUIE_PATH, sortie=SORTIE_SVG)

In [ ]:
graphe_statuts(full_data, finaux)